# 03 — Isolated agent evals (mock)

Every live pipeline agent/node has a sandbox eval task. Isolated runners
open a `document-pipeline` root chain and nest **one** observation so
The-Mailroom can still draw a partial conveyor.

Canonical code:

- [`src/mailroom_sandbox/eval/agents.py`](../src/mailroom_sandbox/eval/agents.py) — `SPECS`
- [`src/mailroom_sandbox/eval/runners.py`](../src/mailroom_sandbox/eval/runners.py) — `run_isolated_eval`
- [`docs/evals.md`](../docs/evals.md)

**Honesty:** mock predictors copy gold labels / fields. `exact_match == 1.0`
proves the harness, scoring sink, and experiment log — **not** model quality.


In [ ]:
import sys
from pathlib import Path

def find_repo_root() -> Path:
    """Walk up from cwd (hostile kernels start in notebooks/)."""
    for cand in [Path.cwd(), *Path.cwd().parents]:
        if (cand / "pyproject.toml").is_file() and (cand / "reports").is_dir():
            return cand
    raise RuntimeError("repo root not found")

ROOT = find_repo_root()
sys.path.insert(0, str(ROOT / "notebooks"))
sys.path.insert(0, str(ROOT / "src"))

from _lib import bootstrap, isolate_outputs

ROOT = bootstrap(ROOT)
OUT = isolate_outputs(ROOT)
print("repo root :", ROOT)
print("sys.path[0]:", sys.path[0])
print("notebook outputs ->", OUT)


## Roster vs retired specialists


In [ ]:
from mailroom_sandbox.eval.agents import SPECS, RETIRED_AGENTS, EVAL_TASKS, spec_for
from mailroom_sandbox.eval import tracing

print("retired (no runners):", RETIRED_AGENTS)
print("composite tasks     :", [t for t in EVAL_TASKS if t not in SPECS])
print()
print(f"{'task':32s} {'observation':24s} type")
for name, spec in SPECS.items():
    print(f"{name:32s} {spec.observation:24s} {tracing.observation_type_for(spec.observation)}")
assert "court_opinions_specialist" not in SPECS
assert "pipeline" in EVAL_TASKS


## Dry-run then mock `judge` / `sorter` / `intake`


In [ ]:
from mailroom_sandbox.eval.runners import run_isolated_eval

plan = run_isolated_eval("judge", mock=True, dry_run=True)
print("dry-run plan:", plan)

judge = run_isolated_eval("judge", mock=True, sample=2, experiment_name="nb03_judge")
sorter = run_isolated_eval("sorter", mock=True, sample=4, experiment_name="nb03_sorter")
intake = run_isolated_eval("intake", mock=True, experiment_name="nb03_intake")
print("judge  scores:", judge["scores"])
print("sorter scores:", sorter["scores"])
print("intake scores:", intake["scores"])
print("sorter observation:", sorter.get("observation") or spec_for("sorter").observation)
assert judge["scores"]["exact_match"] == 1.0
assert sorter["scores"]["n"] == 4


## Reviewer, arbiter, and a specialist extract


In [ ]:
reviewer = run_isolated_eval("sorter_reviewer", mock=True, sample=2, experiment_name="nb03_reviewer")
arbiter = run_isolated_eval("arbiter", mock=True, experiment_name="nb03_arbiter")
contracts = run_isolated_eval(
    "contracts_specialist", mock=True, sample=2, experiment_name="nb03_contracts"
)
print("reviewer :", reviewer["scores"])
print("arbiter  :", arbiter["scores"])
print("contracts:", contracts["scores"])
print("contracts per-row ids:", [r["id"] for r in contracts.get("per_row") or []])
print()
print("CLI equivalent:")
print("  sandbox eval judge --mock")
print("  sandbox eval sorter --mock --sample 4")
print("  sandbox eval contracts_specialist --mock")


## What mock is *not*


In [ ]:
print("HONEST GAP: mock exact_match=1.0 is a harness smoke test.")
print("A local model is `sandbox eval sorter --local` after `sandbox pull-models`.")
print("Vendored mailroom (`sandbox fetch-deps`) is required for live agent classes.")
print("If live_predict raises, runners fall back to mock and set offline_fallback.")
